In [1]:

# ==========================================
# PHASE 3: DATA CLEANING
# ==========================================

# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Locate original datasets

data_path = Path("../data/raw")

# Load original CSV files

drug_reference_raw = pd.read_csv(
    data_path / "drug_reference.csv"
)

patients_raw = pd.read_csv(
    data_path / "patients.csv"
)

prescription_fills_raw = pd.read_csv(
    data_path / "prescription_fills.csv"
)

print("All original datasets loaded successfully!")


All original datasets loaded successfully!


In [2]:

# Create separate working copies of our datasets

drug_reference_clean = drug_reference_raw.copy()

patients_clean = patients_raw.copy()

prescription_fills_clean = prescription_fills_raw.copy()

print("Working copies created successfully!")


Working copies created successfully!


In [3]:

# Record original prescription count

original_fill_count = len(prescription_fills_clean)

# Remove duplicate prescription IDs
# Keep the first occurrence

prescription_fills_clean = (
    prescription_fills_clean
    .drop_duplicates(
        subset="fill_id",
        keep="first"
    )
    .copy()
)

# Calculate updated record count

cleaned_fill_count = len(prescription_fills_clean)

removed_duplicates = original_fill_count - cleaned_fill_count

# Display cleaning summary

print("PRESCRIPTION DEDUPLICATION SUMMARY")
print("=" * 45)

print("Original records:", original_fill_count)

print("Records after deduplication:", cleaned_fill_count)

print("Duplicate records removed:", removed_duplicates)

print(
    "Remaining duplicate fill IDs:",
    prescription_fills_clean["fill_id"].duplicated().sum()
)


PRESCRIPTION DEDUPLICATION SUMMARY
Original records: 25279
Records after deduplication: 25261
Duplicate records removed: 18
Remaining duplicate fill IDs: 0


In [4]:

# Record unique drug names before standardization

reference_before = drug_reference_clean["drug_name"].nunique()

fills_before = prescription_fills_clean["drug_name"].nunique()


# Standardize drug names in drug reference

drug_reference_clean["drug_name"] = (
    drug_reference_clean["drug_name"]
    .astype("string")
    .str.strip()
    .str.casefold()
)


# Standardize drug names in prescription fills

prescription_fills_clean["drug_name"] = (
    prescription_fills_clean["drug_name"]
    .astype("string")
    .str.strip()
    .str.casefold()
)


# Check unique names after standardization

reference_after = drug_reference_clean["drug_name"].nunique()

fills_after = prescription_fills_clean["drug_name"].nunique()


# Display summary

print("DRUG NAME STANDARDIZATION SUMMARY")
print("=" * 45)

print("Reference names before:", reference_before)
print("Reference names after:", reference_after)

print("Prescription names before:", fills_before)
print("Prescription names after:", fills_after)

print("Drug name standardization completed!")


DRUG NAME STANDARDIZATION SUMMARY
Reference names before: 14
Reference names after: 14
Prescription names before: 42
Prescription names after: 14
Drug name standardization completed!


In [5]:

# Check for duplicate drug names in reference

duplicate_reference_names = (
    drug_reference_clean["drug_name"].duplicated().sum()
)

# Compare standardized drug names

reference_set = set(drug_reference_clean["drug_name"])

prescription_set = set(prescription_fills_clean["drug_name"])

unmatched_names = prescription_set - reference_set

# Count matching prescription records

matched_records = prescription_fills_clean[
    prescription_fills_clean["drug_name"].isin(reference_set)
].shape[0]

unmatched_records = len(prescription_fills_clean) - matched_records


# Display validation results only

print("DRUG NAME VALIDATION")
print("=" * 45)

print("Duplicate reference drug names:", duplicate_reference_names)

print("Unmatched unique drug names:", len(unmatched_names))

print("Matching prescription records:", matched_records)

print("Unmatched prescription records:", unmatched_records)


DRUG NAME VALIDATION
Duplicate reference drug names: 0
Unmatched unique drug names: 0
Matching prescription records: 25261
Unmatched prescription records: 0


In [6]:

# Define categorical columns that need standardization

lowercase_columns = {
    "drug_reference": {
        "df": drug_reference_clean,
        "columns": ["drug_class", "measure", "is_generic"]
    },
    "patients": {
        "df": patients_clean,
        "columns": ["plan_type"]
    },
    "prescription_fills": {
        "df": prescription_fills_clean,
        "columns": ["fill_status", "fill_channel"]
    }
}

uppercase_columns = {
    "patients": {
        "df": patients_clean,
        "columns": ["sex", "state"]
    }
}

# Standardize lowercase categories

for dataset_name, details in lowercase_columns.items():

    df = details["df"]

    for col in details["columns"]:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.casefold()
        )

# Standardize uppercase categories

for dataset_name, details in uppercase_columns.items():

    df = details["df"]

    for col in details["columns"]:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.upper()
        )

print("Categorical standardization completed!")


Categorical standardization completed!


In [7]:

# Check missing values after standardization

print("CATEGORICAL STANDARDIZATION VALIDATION")
print("=" * 50)

print(
    "Drug reference missing values:",
    drug_reference_clean.isnull().sum().sum()
)

print(
    "Patient missing values:",
    patients_clean.isnull().sum().sum()
)

print(
    "Prescription fills missing values:",
    prescription_fills_clean.isnull().sum().sum()
)

print("\nCategorical values standardized successfully!")


CATEGORICAL STANDARDIZATION VALIDATION
Drug reference missing values: 0
Patient missing values: 0
Prescription fills missing values: 0

Categorical values standardized successfully!


In [8]:

# ==========================================
# DATE TYPE CONVERSION
# ==========================================

# Convert patient enrollment dates

patients_clean["enrolled_from"] = pd.to_datetime(
    patients_clean["enrolled_from"],
    errors="raise"
)

patients_clean["enrolled_through"] = pd.to_datetime(
    patients_clean["enrolled_through"],
    errors="raise"
)

# Convert prescription fill dates

prescription_fills_clean["filled_on"] = pd.to_datetime(
    prescription_fills_clean["filled_on"],
    errors="raise"
)

print("Date conversion completed successfully!")


Date conversion completed successfully!


In [9]:

# ==========================================
# NUMERICAL TYPE CONVERSION
# ==========================================

# Patient numerical columns

patients_clean["birth_year"] = pd.to_numeric(
    patients_clean["birth_year"],
    errors="raise"
).astype("int64")

# Prescription numerical columns

prescription_fills_clean["fill_id"] = pd.to_numeric(
    prescription_fills_clean["fill_id"],
    errors="raise"
).astype("int64")

prescription_fills_clean["days_supply"] = pd.to_numeric(
    prescription_fills_clean["days_supply"],
    errors="raise"
).astype("int64")

prescription_fills_clean["quantity"] = pd.to_numeric(
    prescription_fills_clean["quantity"],
    errors="raise"
).astype("int64")

prescription_fills_clean["copay_usd"] = pd.to_numeric(
    prescription_fills_clean["copay_usd"],
    errors="raise"
).astype("float64")

print("Numerical conversion completed successfully!")


Numerical conversion completed successfully!


In [10]:

# ==========================================
# DATA TYPE VALIDATION
# ==========================================

print("DRUG REFERENCE DATA TYPES")
print(drug_reference_clean.dtypes)

print("\nPATIENTS DATA TYPES")
print(patients_clean.dtypes)

print("\nPRESCRIPTION FILLS DATA TYPES")
print(prescription_fills_clean.dtypes)


DRUG REFERENCE DATA TYPES
drug_name     string[python]
drug_class    string[python]
measure       string[python]
is_generic    string[python]
dtype: object

PATIENTS DATA TYPES
patient_id                  object
birth_year                   int64
sex                 string[python]
state               string[python]
plan_type           string[python]
enrolled_from       datetime64[ns]
enrolled_through    datetime64[ns]
dtype: object

PRESCRIPTION FILLS DATA TYPES
fill_id                  int64
patient_id              object
filled_on       datetime64[ns]
drug_name       string[python]
days_supply              int64
quantity                 int64
pharmacy_id             object
copay_usd              float64
fill_channel    string[python]
fill_status     string[python]
dtype: object


In [11]:

# Import MySQL connection libraries

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

print("MySQL connection libraries imported!")


MySQL connection libraries imported!


In [12]:

# Enter your MySQL root password securely

db_password = getpass("Enter your MySQL password: ")

# Create MySQL connection URL

connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=db_password,
    host="localhost",
    port=3306,
    database="medication_adherence"
)

# Establish connection

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

# Test connection

with engine.connect() as connection:

    result = connection.execute(
        text("SELECT DATABASE(), VERSION()")
    ).one()

    print("MySQL connection successful!")
    print("Database:", result[0])
    print("MySQL version:", result[1])


Enter your MySQL password:  ········


MySQL connection successful!
Database: medication_adherence
MySQL version: 8.0.40


In [13]:

# Check existing tables in our MySQL database

with engine.connect() as connection:

    result = connection.execute(
        text("SHOW TABLES")
    )

    existing_tables = [row[0] for row in result]

print("EXISTING MYSQL TABLES")
print("=====================")

if existing_tables:
    for table in existing_tables:
        print(table)
else:
    print("No tables found. Database is ready for import.")


EXISTING MYSQL TABLES
No tables found. Database is ready for import.


In [14]:

# ==========================================
# IMPORT CLEANED DATASETS INTO MYSQL
# ==========================================

# Prepare datasets for MySQL import

tables_to_import = {
    "drug_reference": drug_reference_clean,
    "patients": patients_clean,
    "prescription_fills": prescription_fills_clean
}

# Import each dataset into MySQL

for table_name, df in tables_to_import.items():

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="fail",
        index=False,
        chunksize=1000
    )

    print(f"{table_name} imported successfully!")
    print(f"Records imported: {len(df)}")
    print("-" * 40)

print("ALL THREE DATASETS IMPORTED SUCCESSFULLY!")


drug_reference imported successfully!
Records imported: 14
----------------------------------------
patients imported successfully!
Records imported: 2200
----------------------------------------
prescription_fills imported successfully!
Records imported: 25261
----------------------------------------
ALL THREE DATASETS IMPORTED SUCCESSFULLY!


In [15]:

# ==========================================
# PHASE 3: EXPORT CLEANED DATASETS
# ==========================================

# Create processed data folder

output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

# Export cleaned working datasets

drug_reference_clean.to_csv(
    output_path / "drug_reference_clean.csv",
    index=False
)

patients_clean.to_csv(
    output_path / "patients_clean.csv",
    index=False
)

prescription_fills_clean.to_csv(
    output_path / "prescription_fills_clean.csv",
    index=False
)

# Retrieve SQL-joined dataset

prescription_fills_with_measure = pd.read_sql_query(
    "SELECT * FROM prescription_fills_with_measure",
    con=engine
)

prescription_fills_with_measure.to_csv(
    output_path / "prescription_fills_with_measure.csv",
    index=False
)

print("ALL CLEANED DATASETS EXPORTED SUCCESSFULLY!")

print("Drug reference:", len(drug_reference_clean))
print("Patients:", len(patients_clean))
print("Prescription fills:", len(prescription_fills_clean))
print("Joined prescription records:", len(prescription_fills_with_measure))


ALL CLEANED DATASETS EXPORTED SUCCESSFULLY!
Drug reference: 14
Patients: 2200
Prescription fills: 25261
Joined prescription records: 25261
